#### **HOW TO USE THIS PROJECT TO GENERATE SYNTHETIC IMAGES ?**
1. Place this notebook in a separate folder
2. Choose the number of synthetic images you want to generate
3. Specify the prefix for your generated synthetic images (its usually the class name)
4. Specify the **"relative-path"** to the generator for that particular class
5. Run all cells
6. Your generated images should be stored at **"./generated_images/"**

In [1]:
num_images = 1991
class_name = 'AD'
path_to_generator = '../synthetic_localised/AD/best_gen/MoI_gen_422_(5.89).h30'

#### **IMPORTING NECESSARY DEPENDENCIES**

In [2]:
import os
import cv2
import shutil
import numpy as np
import seaborn as sns
from tqdm import tqdm
from numpy import cov
from PIL import Image
from numpy import trace
import tensorflow as tf
from numpy import asarray
from tensorflow import keras
from scipy.linalg import sqrtm
from numpy import iscomplexobj
import matplotlib.pyplot as plt
from numpy.random import randint
from tensorflow.keras import layers
from skimage.transform import resize
from IPython.display import FileLink
from keras.datasets.mnist import load_data
from keras.applications.inception_v3 import InceptionV3
from skimage.metrics import structural_similarity as ssim
from keras.applications.inception_v3 import preprocess_input
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

2025-03-23 22:29:54.193242: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-23 22:29:54.358715: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-23 22:29:55.035932: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/conda/lib/python3.10/site-packages/cv2/../../lib64:/usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/lib/x86_64-linux-gnu/:/opt/conda/lib
2025-03-23 22:29:55.036079: W tensor

#### **DEFINING IMPORTANT VARIABLES AND PATHS**

In [3]:
BATCH_SIZE = 32
noise_dim = 256
working_directory = '../synthetic_localised/generated_images/'
os.makedirs(working_directory, exist_ok=True)
generator = tf.keras.models.load_model(path_to_generator)

2025-03-23 22:29:55.815398: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-23 22:29:55.843731: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-23 22:29:55.844557: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-23 22:29:55.845653: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags

#### **DEFINING IMPORTANT FUNCTIONS**

In [4]:
def clear_workspace():
    for filename in os.listdir(working_directory):
        file_path = os.path.join(working_directory, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print('Failed to delete %s. Reason: %s' % (file_path, e))

In [5]:
def download_synthetic_images(generator, number_of_images, class_name='None'):
    clear_workspace()
    
    # Define the working directory and images folder
    folder_path = os.path.join(working_directory, class_name)
    os.makedirs(folder_path, exist_ok=True)

    # Get existing images count
    existing_images = len(os.listdir(folder_path))
    
    # Initialize tqdm with existing count
    pbar = tqdm(total=number_of_images, initial=existing_images, position=0, leave=True)

    counter = existing_images + 1  # Continue numbering from the last saved image

    while counter <= number_of_images:
        remaining_images = number_of_images - existing_images
        batch_size = min(BATCH_SIZE, remaining_images)  # Avoid generating excess images

        # Generate synthetic images
        random_latent_vectors = tf.random.normal(shape=(batch_size, noise_dim))
        synthetic_images_batch = generator(random_latent_vectors).numpy()

        for image in synthetic_images_batch:
            if counter > number_of_images:  # Stop early if the required count is reached
                break  

            # Normalize and convert to uint8
            image = ((image * 127.5) + 127.5).astype(np.uint8)

            # Define image save path with .png extension
            image_path = os.path.join(folder_path, f"{class_name} ({counter}).png")

            if cv2.imwrite(image_path, image):
                pbar.update(1)  # Update progress bar only on successful save
                counter += 1
            else:
                print(f"Failed to save: {image_path}")

    pbar.close()  # Ensure tqdm closes properly
    return 'GENERATED AND SAVED ALL IMAGES'

#### **GENERATING AND SAVING SYNTHETIC IMAGES**

In [6]:
download_synthetic_images(generator, number_of_images = num_images, class_name = class_name)

100%|█████████████████████████████████████| 1991/1991 [00:11<00:00, 179.19it/s]


'GENERATED AND SAVED ALL IMAGES'